# **Demo 02: Executing Serial and Parallel Workflows Using LangGraph**


**Objective:** To demonstrate how to build structured AI workflows using LangGraph's `StateGraph`. This demo shows how to execute tasks **serially** (one node feeds the next) and **in parallel** (multiple nodes run concurrently using the `Send` API), and how to fan results back in to a synthesising node.

**Prerequisites:** OpenAI API key (stored in `.env` as `OPENAI_API_KEY`)

**Tools required:** Python

**Scenario:** A healthcare AI platform needs to research and validate the latest AI advancements. Two workflows are required: a **serial pipeline** where a researcher feeds a validator who feeds a writer (each step depends on the last), and a **parallel pipeline** where two independent research branches run simultaneously and are synthesised into a single executive report — cutting total latency roughly in half.

In [ ]:
# Step 1: Install required libraries
#
# langgraph        — the graph execution framework; provides StateGraph, Send, and graph compilation
# langchain-openai — LangChain's OpenAI integration; provides ChatOpenAI
# langchain-core   — base message types and LangChain primitives (HumanMessage, etc.)
# python-dotenv    — loads key=value pairs from a .env file into os.environ

!pip install langgraph langchain-openai langchain-core python-dotenv

In [15]:
# Step 2: Imports and OpenAI LLM setup
#
# operator   — provides operator.add, used as a reducer to merge branch lists in parallel state
# TypedDict  — defines graph state schemas as typed dicts; each key becomes a state field
# Annotated  — attaches metadata to type hints; here it carries the reducer function
# List       — standard list type used in the parallel state definition

import operator
from typing import TypedDict, Annotated, List

from dotenv import load_dotenv                     # reads key=value pairs from a .env file into os.environ
from langchain_openai import ChatOpenAI            # LangChain wrapper around OpenAI's chat models
from langchain_core.messages import HumanMessage   # wraps a user-turn message for the chat API
from langgraph.graph import StateGraph, START, END # StateGraph builds the graph; START/END are sentinel nodes
from langgraph.types import Send                   # Send dispatches a node call with custom per-branch state

# ── Load credentials and initialise the LLM ──────────────────────────────────
# load_dotenv() looks for a .env file in the current directory and loads its
# key=value pairs into os.environ. OPENAI_API_KEY is picked up automatically
# by ChatOpenAI without needing to reference it explicitly.
load_dotenv()  # loads OPENAI_API_KEY from .env

# ChatOpenAI wraps GPT-4o-mini.
# temperature=0.3 keeps outputs focused and reproducible while allowing some variety.
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.3,
)

print("LLM ready: gpt-4o-mini")


LLM ready: gpt-4o-mini


---
## Part 1: Serial Execution

In a **serial workflow**, each node completes before the next begins. Node output is passed through shared state:

```
START → Researcher → Validator → Writer → END
```

Use this pattern when each step depends on the output of the previous one.

In [16]:
# Step 3: Define state and nodes for the serial workflow
#
# KEY CONCEPT — State is the shared memory of the graph:
#   Every node receives the full state dict and returns a partial dict with
#   only the keys it updates. LangGraph merges the partial update back into
#   the state before passing it to the next node.

class SerialState(TypedDict):
    query: str            # the original user question; set once at invocation, read by all nodes
    research_notes: str   # raw notes produced by researcher_node
    validated_notes: str  # fact-checked version produced by validator_node
    final_report: str     # executive summary produced by writer_node


def researcher_node(state: SerialState) -> dict:
    """Node 1: Research the query and produce raw notes."""
    # Build the prompt using the incoming query from shared state.
    prompt = (
        f"You are a healthcare AI researcher. Research this topic and provide "
        f"detailed notes on 3 key advancements (4-5 sentences each):\n{state['query']}"
    )
    response = llm.invoke([HumanMessage(content=prompt)])
    # Return only the key(s) this node updates; LangGraph merges them into state.
    return {"research_notes": response.content}


def validator_node(state: SerialState) -> dict:
    """Node 2: Validate and fact-check the research notes."""
    # Reads research_notes written by researcher_node — strict sequential dependency.
    prompt = (
        f"You are a medical fact-checker. Review these AI healthcare research notes "
        f"and flag any inaccuracies or unsupported claims. Return a corrected version:\n"
        f"{state['research_notes']}"
    )
    response = llm.invoke([HumanMessage(content=prompt)])
    return {"validated_notes": response.content}


def writer_node(state: SerialState) -> dict:
    """Node 3: Write an executive summary from the validated notes."""
    # Reads validated_notes written by validator_node — depends on both prior nodes.
    prompt = (
        f"You are a technical writer. Using these validated research notes, "
        f"write a concise executive summary (max 200 words, bullet points):\n"
        f"{state['validated_notes']}"
    )
    response = llm.invoke([HumanMessage(content=prompt)])
    return {"final_report": response.content}


print("Serial nodes defined.")


Serial nodes defined.


In [17]:
# Step 4: Build and compile the serial graph
#
# StateGraph(SerialState) declares the graph and its shared state schema.
# Nodes are registered with add_node(); edges declare execution order.
# compile() validates the graph structure and returns a runnable object.

serial_builder = StateGraph(SerialState)

# Register each node: the string name is used in add_edge() to wire the graph.
serial_builder.add_node("researcher", researcher_node)
serial_builder.add_node("validator",  validator_node)
serial_builder.add_node("writer",     writer_node)

# add_edge(A, B) means: after A completes, run B next.
# This creates a strict sequential chain — each node must finish before the next starts.
serial_builder.add_edge(START,        "researcher")  # START is the graph entry point
serial_builder.add_edge("researcher", "validator")
serial_builder.add_edge("validator",  "writer")
serial_builder.add_edge("writer",     END)           # END signals the graph has finished

# compile() checks for disconnected nodes, unreachable END, and other structural errors.
serial_graph = serial_builder.compile()
print("Serial graph compiled.")


Serial graph compiled.


In [18]:
# Step 5: Run the serial workflow
#
# invoke() is synchronous and blocks until all nodes have completed.
# The input dict must supply every key that has no default value;
# here only 'query' is needed — the other keys are written by the nodes.

serial_input = {"query": "Latest advancements in AI for healthcare diagnostics"}

print("=== SERIAL EXECUTION ===")
print(f"Query: {serial_input['query']}\n")

# invoke() triggers: START → researcher → validator → writer → END
# Each node reads from and writes to the shared SerialState.
serial_result = serial_graph.invoke(serial_input)

# Print a preview of each state key to trace how data flowed through the chain.
print("[Research Notes]:\n", serial_result["research_notes"][:300], "...\n")
print("[Validated Notes]:\n", serial_result["validated_notes"][:300], "...\n")
print("[Final Report]:\n", serial_result["final_report"])


=== SERIAL EXECUTION ===
Query: Latest advancements in AI for healthcare diagnostics

[Research Notes]:
 Here are three key advancements in AI for healthcare diagnostics:

1. **Deep Learning in Medical Imaging**: Recent advancements in deep learning algorithms have significantly improved the accuracy of medical imaging diagnostics. Convolutional neural networks (CNNs) are now being utilized to analyze  ...

[Validated Notes]:
 Here is a corrected version of the AI healthcare research notes, with inaccuracies and unsupported claims flagged and addressed:

---

Here are three key advancements in AI for healthcare diagnostics:

1. **Deep Learning in Medical Imaging**: Recent advancements in deep learning algorithms have sign ...

[Final Report]:
 ### Executive Summary: Key Advancements in AI for Healthcare Diagnostics

- **Deep Learning in Medical Imaging**: 
  - Utilization of Convolutional Neural Networks (CNNs) enhances accuracy in analyzing X-rays, MRIs, and CT scans.
  - AI can detec

---
## Part 2: Parallel Execution

In a **parallel workflow**, multiple nodes run concurrently using LangGraph's `Send` API. A router fans out to independent branches; results are aggregated with a reducer (`Annotated[List[str], operator.add]`) and merged in a synthesiser node:

```
              ┌─► research_branch (topic A) ─┐
START → router─┤                              ├─► synthesiser → END
              └─► research_branch (topic B) ─┘
```

Use this pattern when branches are **independent** and you want to minimise total latency.

In [19]:
# Step 6: Define state and nodes for the parallel workflow
#
# KEY CONCEPT — Reducers merge concurrent branch outputs:
#   When multiple branches write to the same state key at the same time,
#   LangGraph needs a reducer to combine their outputs without overwriting
#   each other. Annotated[List[str], operator.add] tells LangGraph:
#   "append lists from each branch together". Without this, only the last
#   branch's value would survive.

class ParallelState(TypedDict):
    query: str
    branch_results: Annotated[List[str], operator.add]  # reducer: each branch appends one item
    final_report: str


class BranchState(TypedDict):
    """State passed to each individual research branch via Send.

    BranchState is intentionally minimal — it carries only the data
    each branch needs, keeping the branches fully independent.
    """
    topic: str


def router_node(state: ParallelState):
    """
    Fan-out node: returns a list of Send() calls, one per branch.
    LangGraph executes all Send() targets concurrently.

    Send(node_name, state) schedules node_name to run with the given
    per-branch state. Each Send runs in its own execution context,
    isolated from the other branches.
    """
    # Define two independent research topics derived from the parent query.
    topics = [
        f"AI in medical imaging and diagnostics — {state['query']}",
        f"AI in drug discovery and personalised medicine — {state['query']}",
    ]
    # Return a Send for each topic. LangGraph fans out and runs them in parallel.
    return [Send("research_branch", {"topic": t}) for t in topics]


def research_branch(state: BranchState) -> dict:
    """
    Each branch independently researches its assigned topic.
    Returns a single-item list; the reducer concatenates across branches.
    """
    prompt = (
        f"In 4-5 sentences, summarise the latest AI advancements in: {state['topic']}"
    )
    response = llm.invoke([HumanMessage(content=prompt)])
    # Wrap in a list so operator.add can concatenate results from every branch.
    return {"branch_results": [response.content]}


def synthesiser_node(state: ParallelState) -> dict:
    """Fan-in node: combines all branch results into one executive report.

    By the time this node runs, branch_results contains the concatenated
    outputs from every branch (thanks to the operator.add reducer).
    """
    # Join all branch outputs into a single numbered block for the LLM to synthesise.
    combined = "\n\n".join(
        f"Branch {i+1}:\n{r}" for i, r in enumerate(state["branch_results"])
    )
    prompt = (
        f"You are a technical writer. Synthesise the following parallel research results "
        f"into a single executive summary (max 250 words, bullet points):\n\n{combined}"
    )
    response = llm.invoke([HumanMessage(content=prompt)])
    return {"final_report": response.content}


print("Parallel nodes defined.")


Parallel nodes defined.


In [20]:
# Step 7: Build and compile the parallel graph
#
# The parallel graph differs from the serial graph in two key ways:
#   1. add_conditional_edges on START — router_node returns Send() objects
#      which LangGraph uses to fan out to multiple 'research_branch' runs.
#   2. All branches share a single outgoing edge to 'synthesiser', which
#      acts as the fan-in point once every branch has finished.

parallel_builder = StateGraph(ParallelState)

# Register the worker node and the fan-in node.
# Note: router_node is NOT added here — it is used as the routing
# function in add_conditional_edges, not as a graph node itself.
parallel_builder.add_node("research_branch", research_branch)
parallel_builder.add_node("synthesiser",     synthesiser_node)

# add_conditional_edges(source, routing_fn, targets):
#   routing_fn is called with the current state and returns either a node
#   name or a list of Send() objects. The third argument lists valid target
#   node names so LangGraph can validate the routing at compile time.
parallel_builder.add_conditional_edges(START, router_node, ["research_branch"])

# Every branch converges on the synthesiser once it finishes.
# The operator.add reducer merges branch_results before synthesiser_node runs.
parallel_builder.add_edge("research_branch", "synthesiser")
parallel_builder.add_edge("synthesiser",     END)

parallel_graph = parallel_builder.compile()
print("Parallel graph compiled.")


Parallel graph compiled.


In [21]:
# Step 8: Run the parallel workflow
#
# branch_results must be initialised as an empty list so the reducer
# has a starting value to append to. The other keys are populated by the graph.

parallel_input = {"query": "Recent breakthroughs in healthcare AI", "branch_results": []}

print("=== PARALLEL EXECUTION ===")
print(f"Query: {parallel_input['query']}\n")

# invoke() triggers: START → router_node → [branch A, branch B] → synthesiser → END
# Branches A and B run concurrently; invoke() waits for both before continuing.
parallel_result = parallel_graph.invoke(parallel_input)

# Print a preview of each branch result and the final synthesised report.
print("[Branch Results]:")
for i, r in enumerate(parallel_result["branch_results"], 1):
    print(f"\n  Branch {i}:\n  {r[:200]}...")

print("\n[Synthesised Report]:\n", parallel_result["final_report"])


=== PARALLEL EXECUTION ===
Query: Recent breakthroughs in healthcare AI

[Branch Results]:

  Branch 1:
  Recent advancements in AI for medical imaging and diagnostics have significantly enhanced the accuracy and efficiency of disease detection. Machine learning algorithms are now capable of analyzing com...

  Branch 2:
  Recent advancements in AI for drug discovery and personalized medicine have significantly accelerated the development of new therapeutics and tailored treatment plans. Machine learning algorithms are ...

[Synthesised Report]:
 **Executive Summary: Advancements in AI for Healthcare**

- **Enhanced Medical Imaging and Diagnostics**: 
  - AI technologies have improved the accuracy and efficiency of disease detection in medical imaging.
  - Machine learning algorithms analyze complex data from MRI and CT scans to identify conditions like tumors and fractures with high precision.
  - Breakthroughs include AI systems that predict patient outcomes and assist radiologists i

---
## Summary

This demo showed two fundamental LangGraph workflow patterns:

**Serial execution** — nodes chained with `add_edge`. Each node's output is written to shared `State` and read by the next node. Use when steps have strict dependencies (research → validate → write).

**Parallel execution** — a router returns a list of `Send()` calls, each targeting the same node with different input. LangGraph runs the branches concurrently. Results are merged back into shared state using an `Annotated` reducer (`operator.add` for lists). A fan-in node then synthesises the combined results.

**Key classes used:**
- `StateGraph` — defines the graph and its shared state schema
- `Send` — passes per-branch state to a node for concurrent execution
- `Annotated[List[str], operator.add]` — automatically merges branch outputs into one list
- `add_conditional_edges` — routes dynamically based on a function's return value